In [ ]:
import ee
import pandas as pd

ee.Authenticate()
try:
    ee.Initialize(project='ee-sakda-451407')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='ee-sakda-451407')

In [ ]:
# ---- config ----
# ใช้ FAO GAUL 2015 ชุดเดียวกับตอน train
# ใช้ FIRMS T21 (MODIS) เหมือนตอน train เป๊ะๆ
# ช่วงเวลา: ต.ค. 2025 – เม.ย. 2026 (7 เดือน)

START = '2025-10-01'
END   = '2026-04-30'

GAUL = ee.FeatureCollection('FAO/GAUL/2015/level1') \
    .filter(ee.Filter.eq('ADM0_NAME', 'Thailand'))

PROVINCES = {
    'Mae Hong Son': GAUL.filter(ee.Filter.eq('ADM1_NAME', 'Mae Hong Son')),
    'Nan':          GAUL.filter(ee.Filter.eq('ADM1_NAME', 'Nan')),
    'Uttaradit':    GAUL.filter(ee.Filter.eq('ADM1_NAME', 'Uttaradit')),
}

# ตรวจสอบชื่อจังหวัดใน GAUL ก่อน
for name, fc in PROVINCES.items():
    print(f'{name}: {fc.size().getInfo()} features')

In [ ]:
# ---- function: MODIS FIRMS T21 monthly hotspot count ----
# ตรงกับวิธีนับที่ใช้ตอน train โมเดลเป๊ะๆ

def build_year_months(start, end):
    ym = []
    y, m = int(start[:4]), int(start[5:7])
    ey, em = int(end[:4]), int(end[5:7])
    while (y < ey) or (y == ey and m <= em):
        ym.append(y * 100 + m)
        m += 1
        if m > 12:
            m = 1
            y += 1
    return ym


def get_monthly_hotspots(start_date, end_date, study_area):
    firms = ee.ImageCollection('FIRMS') \
        .filterDate(start_date, end_date) \
        .filterBounds(study_area)

    size = firms.size().getInfo()
    print(f'  FIRMS raw images: {size}')
    if size == 0:
        return None

    def to_presence(img):
        hp = img.select('T21').gt(0).rename('hotspot_presence')
        return img.addBands(hp).copyProperties(img, ['system:time_start'])

    firms_p = firms.map(to_presence)
    year_months = build_year_months(start_date, end_date)

    def make_monthly(ym):
        yr  = ee.Number(ym).divide(100).floor().int()
        mo  = ee.Number(ym).mod(100).int()
        st  = ee.Date.fromYMD(yr, mo, 1)
        en  = st.advance(1, 'month')
        col = firms_p.filterDate(st, en)
        cnt = col.size()
        val = ee.Number(
            ee.Algorithms.If(
                cnt.gt(0),
                ee.Number(
                    col.select('hotspot_presence').sum().reduceRegion(
                        reducer=ee.Reducer.sum(),
                        geometry=study_area,
                        scale=1000,
                        maxPixels=1e9
                    ).get('hotspot_presence', 0)
                ),
                0
            )
        )
        return ee.Image.constant(val).rename('hotspot_count') \
            .set('system:time_start', st.millis()) \
            .set('system:index', st.format('YYYY_MM')) \
            .set('year', yr).set('month', mo)

    ic = ee.ImageCollection.fromImages(
        ee.List(year_months).map(make_monthly)
    )
    return ic.select(['hotspot_count'])


def ic_to_series(ic, study_area):
    def to_feat(img):
        val = img.reduceRegion(
            reducer=ee.Reducer.first(),
            geometry=study_area,
            scale=1000, maxPixels=1e9
        ).get('hotspot_count', 0)
        return ee.Feature(None, {
            'date':          ee.Date(img.get('system:time_start')).format('YYYY-MM'),
            'year':          img.get('year'),
            'month':         img.get('month'),
            'hotspot_count': val
        })
    rows = [
        f['properties']
        for f in ee.FeatureCollection(ic.map(to_feat)).getInfo()['features']
    ]
    df = pd.DataFrame(rows).sort_values(['year', 'month']).reset_index(drop=True)
    return df

In [ ]:
# ---- ดึงข้อมูลทีละจังหวัด ----

series = {}
for prov, sa in PROVINCES.items():
    print(f'\n=== {prov} ===')
    ic = get_monthly_hotspots(START, END, sa)
    if ic is None:
        print('  no data')
        series[prov] = {}
    else:
        df_p = ic_to_series(ic, sa)
        print(df_p[['date', 'hotspot_count']].to_string(index=False))
        series[prov] = dict(zip(df_p['date'], df_p['hotspot_count']))

In [ ]:
# ---- สร้างตาราง wide-format และ export ----
# คอลัมน์: Month | Mae Hong Son | Nan | Uttaradit

all_dates = sorted({d for r in series.values() for d in r})
wide = pd.DataFrame({'Month': all_dates})
for prov in PROVINCES:
    wide[prov] = wide['Month'].map(series.get(prov, {}))

print('=== Observed Hotspot (Oct 2025 – Apr 2026) ===')
print(wide.to_string(index=False))

wide.to_csv('observed_hotspot_3provinces.csv', index=False)
wide.to_excel('observed_hotspot_3provinces.xlsx', index=False, sheet_name='Observed_Hotspot')
print('\nSaved: observed_hotspot_3provinces.csv / .xlsx')